**Lab5_Ex_4_By_Prajakta Ghorpade SCFP125004**


Exercise 4 — Outdoor Event Plan to Weather-Adjusted Recommendation Pipeline
Build a 3-step pipeline. Step 1 extracts the event details (event type, planned date, location, indoor/outdoor preference) from a free-form event description. Step 2 calls a weather API for the extracted location and date, and normalizes the response into structured weather data. Step 3 generates a recommendation message for the event organizer (proceed as planned / reschedule / move indoors) that combines both the event details and the weather data.

Input: event_description_text

Output: the structured event details from Step 1, the structured weather data from Step 2, and the recommendation message from Step 3

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: event_type, event_date, location, indoor_outdoor_preference)

Step 2 — Template: PromptTemplate (used to format the weather API call/query from Step 1's fields; the API response itself is normalized into structured output, not freely generated by the LLM)  |  Parser: PydanticOutputParser (fields: temperature, precipitation_chance, conditions)

Step 3 — Template: ChatPromptTemplate (system persona: "event planning assistant")  |  Parser: StrOutputParser



Install required libraries

In [1]:
!pip install -q langchain-groq langchain-core pydantic requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00


Import libraries and initialize the LLM

In [2]:
import requests
from datetime import date
from typing import Optional

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

llm.invoke("hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.063157237, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.019108387, 'prompt_tokens_details': None, 'queue_time': 0.157313174, 'total_time': 0.082265624}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e11c8bbf69', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1675f-bad7-77e2-809b-a3d8db8391a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129, 'output_token_details': {'reasoning': 39}})

Extract event details

In [3]:
class EventDetails(BaseModel):
    event_type: str = Field(description="Type of event")
    event_date: str = Field(description="Event date in YYYY-MM-DD format")
    location: str = Field(description="Event location")
    indoor_outdoor_preference: str = Field(
        description="Indoor, outdoor, or flexible"
    )


event_parser = PydanticOutputParser(pydantic_object=EventDetails)

event_prompt = PromptTemplate(
    template="""
Extract event details from the following description.

If a year is not specified, use the current year only when the
intended date is clear. Do not invent a date or location; use
"Not provided" if missing.

Description:
{event_description_text}

{format_instructions}
""",
    input_variables=["event_description_text"],
    partial_variables={
        "format_instructions": event_parser.get_format_instructions()
    }
)

event_chain = event_prompt | llm | event_parser

Fetch and normalize weather data

In [4]:
class WeatherData(BaseModel):
    temperature: Optional[float] = Field(
        description="Temperature in Celsius; null if unavailable"
    )
    precipitation_chance: Optional[float] = Field(
        description="Precipitation probability in percent; null if unavailable"
    )
    conditions: str = Field(
        description="Weather conditions or an error message"
    )


weather_parser = PydanticOutputParser(pydantic_object=WeatherData)

weather_prompt = PromptTemplate(
    template="""
Prepare a weather API query using these event details.

Location: {location}
Date: {event_date}

Use the location for geocoding and the date for the weather forecast.
Return the location and date needed by the API.
""",
    input_variables=["location", "event_date"]
)


def get_weather(event):
    query = weather_prompt.format(
        location=event.location,
        event_date=event.event_date
    )

    if event.location == "Not provided":
        return weather_parser.parse(
            '{"temperature": null, "precipitation_chance": null, '
            '"conditions": "Weather unavailable: location not provided"}'
        )

    try:
        # Convert location to latitude and longitude.
        geo_response = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": event.location, "count": 1},
            timeout=15
        )
        geo_response.raise_for_status()
        places = geo_response.json().get("results", [])

        if not places:
            return weather_parser.parse(
                '{"temperature": null, "precipitation_chance": null, '
                '"conditions": "Weather unavailable: location not found"}'
            )

        place = places[0]
        event_day = date.fromisoformat(event.event_date)

        # Historical dates use archive data; future dates use forecast data.
        if event_day < date.today():
            api_url = "https://archive-api.open-meteo.com/v1/archive"
        else:
            api_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            api_url,
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "start_date": event.event_date,
                "end_date": event.event_date,
                "daily": "temperature_2m_mean,precipitation_probability_max,weather_code",
                "timezone": "auto"
            },
            timeout=20
        )
        weather_response.raise_for_status()
        data = weather_response.json().get("daily", {})

        if not data.get("time"):
            return weather_parser.parse(
                '{"temperature": null, "precipitation_chance": null, '
                '"conditions": "Weather unavailable for the selected date"}'
            )

        temp = data.get("temperature_2m_mean", [None])[0]
        rain = data.get("precipitation_probability_max", [None])[0]
        code = data.get("weather_code", [None])[0]

        conditions_map = {
            0: "Clear sky",
            1: "Mainly clear",
            2: "Partly cloudy",
            3: "Overcast",
            45: "Fog",
            48: "Depositing rime fog",
            51: "Light drizzle",
            53: "Moderate drizzle",
            55: "Dense drizzle",
            61: "Light rain",
            63: "Moderate rain",
            65: "Heavy rain",
            71: "Light snow",
            73: "Moderate snow",
            75: "Heavy snow",
            80: "Rain showers",
            81: "Moderate rain showers",
            82: "Violent rain showers",
            95: "Thunderstorm"
        }

        # Normalize the API response into the required Pydantic fields.
        normalized = {
            "temperature": temp,
            "precipitation_chance": rain,
            "conditions": conditions_map.get(code, "Unknown conditions")
        }

        return weather_parser.parse(
            WeatherData(**normalized).model_dump_json()
        )

    except (requests.RequestException, ValueError, KeyError, IndexError) as e:
        return WeatherData(
            temperature=None,
            precipitation_chance=None,
            conditions=f"Weather unavailable: {str(e)}"
        )


weather_chain = weather_prompt

Generate the recommendation

In [5]:
recommendation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an event planning assistant. Give a clear, practical "
        "recommendation: proceed as planned, reschedule, or move indoors. "
        "Base the recommendation only on the supplied event details and "
        "weather data. If weather data is unavailable, do not guess; explain "
        "that the organizer should check a reliable forecast before deciding."
    ),
    (
        "human",
        """
Event details:
{event_details}

Weather data:
{weather_data}

Provide the recommendation and briefly explain why.
"""
    )
])

recommendation_chain = recommendation_prompt | llm | StrOutputParser()

complete 3-step pipeline

In [6]:
event_description_text = """
We are planning an outdoor college cultural festival in Solapur
on 2026-10-30. The event will include dance performances and
student activities. We prefer an outdoor venue.
"""

# Step 1: Extract event details
event_details = event_chain.invoke({
    "event_description_text": event_description_text
})

# Step 2: Fetch real weather data
weather_data = get_weather(event_details)

# Step 3: Generate recommendation
if weather_data.temperature is None and "unavailable" in weather_data.conditions.lower():
    recommendation = (
        "Weather-based recommendation unavailable. "
        "Please check a reliable forecast before deciding whether to "
        "proceed, reschedule, or move indoors."
    )
else:
    recommendation = recommendation_chain.invoke({
        "event_details": event_details.model_dump_json(),
        "weather_data": weather_data.model_dump_json()
    })

# Display the complete output
print("STEP 1 — STRUCTURED EVENT DETAILS")
print(event_details.model_dump_json(indent=2))

print("\nSTEP 2 — STRUCTURED WEATHER DATA")
print(weather_data.model_dump_json(indent=2))

print("\nSTEP 3 — RECOMMENDATION")
print(recommendation)

STEP 1 — STRUCTURED EVENT DETAILS
{
  "event_type": "college cultural festival",
  "event_date": "2026-10-30",
  "location": "Solapur",
  "indoor_outdoor_preference": "outdoor"
}

STEP 2 — STRUCTURED WEATHER DATA
{
  "temperature": 27.6,
  "precipitation_chance": 18.0,
  "conditions": "Clear sky"
}

STEP 3 — RECOMMENDATION
**Recommendation:** Proceed as planned (outdoor).

**Why:**  
- The forecast shows a clear sky with only an 18 % chance of precipitation, which is low enough that rain is unlikely to disrupt the event.  
- The temperature (≈28 °C) is comfortable for outdoor activities.  
- Since the event is a college cultural festival and the organizer prefers an outdoor setting, the current weather conditions support that preference.
